# DataScienceUtils - Outlier Detection Test

Testing outlier detection methods with Python bindings.

In [ ]:
import numpy as np
import datascienceutils as dsu
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
print(f"DataScienceUtils version: {dsu.__version__}")

## Generate Test Data with Outliers

In [ ]:
# Create data with outliers
np.random.seed(42)
normal_data = np.random.normal(50, 10, 100)
outliers = np.array([5, 10, 90, 95, 100])  # Add some outliers
data = np.concatenate([normal_data, outliers])

print(f"Data shape: {data.shape}")
print(f"Mean: {data.mean():.2f}")
print(f"Std: {data.std():.2f}")
print(f"Min: {data.min():.2f}, Max: {data.max():.2f}")

In [ ]:
# Visualize original data
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(data, bins=30, edgecolor='black', alpha=0.7)
plt.title('Histogram with Outliers')
plt.xlabel('Value')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.boxplot(data, vert=True)
plt.title('Box Plot')
plt.ylabel('Value')

plt.tight_layout()
plt.show()

## 1. Sigma Deviation Method

In [ ]:
outlier_indices, lower, upper = dsu.detect_outliers_sigma(data, 3.0)

print(f"Sigma Deviation Method (3σ):")
print(f"  Bounds: [{lower:.2f}, {upper:.2f}]")
print(f"  Found {len(outlier_indices)} outliers")
print(f"  Outlier values: {data[outlier_indices]}")

## 2. IQR Method

In [ ]:
outlier_indices_iqr, lower_iqr, upper_iqr = dsu.detect_outliers_iqr(data, 1.5)

print(f"IQR Method (k=1.5):")
print(f"  Bounds: [{lower_iqr:.2f}, {upper_iqr:.2f}]")
print(f"  Found {len(outlier_indices_iqr)} outliers")
print(f"  Outlier values: {data[outlier_indices_iqr]}")

## 3. Z-Score Method

In [ ]:
outlier_indices_z = dsu.detect_outliers_zscore(data, 3.0)

print(f"Z-Score Method (threshold=3.0):")
print(f"  Found {len(outlier_indices_z)} outliers")
print(f"  Outlier values: {data[outlier_indices_z]}")

## 4. Modified Z-Score Method (MAD-based)

In [ ]:
outlier_indices_mad = dsu.detect_outliers_modified_zscore(data, 3.5)

print(f"Modified Z-Score Method (threshold=3.5):")
print(f"  Found {len(outlier_indices_mad)} outliers")
print(f"  Outlier values: {data[outlier_indices_mad]}")

## 5. Comparison of Methods

In [ ]:
# Compare all methods
methods = {
    'Sigma (3σ)': outlier_indices,
    'IQR (k=1.5)': outlier_indices_iqr,
    'Z-Score': outlier_indices_z,
    'Modified Z-Score': outlier_indices_mad
}

print("\nComparison of Outlier Detection Methods:")
print("=" * 50)
for method, indices in methods.items():
    print(f"{method:20s}: {len(indices):3d} outliers")

In [ ]:
# Visualize outliers detected by each method
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, (method, outlier_idx) in enumerate(methods.items()):
    ax = axes[idx]
    
    # Create mask for outliers
    is_outlier = np.zeros(len(data), dtype=bool)
    is_outlier[outlier_idx] = True
    
    # Plot
    ax.scatter(range(len(data)), data, c=is_outlier, cmap='RdYlGn_r', 
               alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    ax.set_title(f'{method}\n({len(outlier_idx)} outliers detected)')
    ax.set_xlabel('Index')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Outlier Removal

In [ ]:
# Remove outliers using IQR method
cleaned_data = dsu.remove_outliers(data, outlier_indices_iqr)

print(f"Original data: {len(data)} points")
print(f"Cleaned data: {len(cleaned_data)} points")
print(f"Removed: {len(data) - len(cleaned_data)} points")
print(f"\nCleaned data statistics:")
print(f"  Mean: {cleaned_data.mean():.2f}")
print(f"  Std: {cleaned_data.std():.2f}")
print(f"  Min: {cleaned_data.min():.2f}, Max: {cleaned_data.max():.2f}")

In [ ]:
# Compare before and after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(data, bins=30, edgecolor='black', alpha=0.7, color='red')
axes[0].set_title('Before Outlier Removal')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

axes[1].hist(cleaned_data, bins=30, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('After Outlier Removal')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 7. Percentile Capping

In [ ]:
# Cap outliers at 5th and 95th percentiles
capped_data = dsu.cap_outliers_percentile(data, 5.0, 95.0)

print(f"Percentile Capping (5th-95th):")
print(f"  Original range: [{data.min():.2f}, {data.max():.2f}]")
print(f"  Capped range: [{capped_data.min():.2f}, {capped_data.max():.2f}]")
print(f"  Values capped: {np.sum(data != capped_data)}")

In [ ]:
# Visualize capping
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(range(len(data)), data, alpha=0.6, label='Original')
plt.scatter(range(len(capped_data)), capped_data, alpha=0.6, label='Capped', marker='x')
plt.title('Percentile Capping Effect')
plt.xlabel('Index')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot([data, capped_data], labels=['Original', 'Capped'])
plt.title('Box Plot Comparison')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

All outlier detection methods are working correctly! The Rust implementation provides:
- Multiple detection strategies
- Fast computation
- Flexible outlier handling (removal vs capping)